In [20]:
import pandas as pd

# 1. Load the dataset into a Pandas DataFrame
df = pd.read_csv('owid-covid-data.csv')

# 2. Get Number of records (rows) and variables (columns)
print(f"Number of Records: {df.shape[0]}")
print(f"Number of Variables: {df.shape[1]}")

# 3. Get Data Types
df.info()

Number of Records: 350085
Number of Variables: 67
<class 'pandas.DataFrame'>
RangeIndex: 350085 entries, 0 to 350084
Data columns (total 67 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   iso_code                                    350085 non-null  str    
 1   continent                                   333420 non-null  str    
 2   location                                    350085 non-null  str    
 3   date                                        350085 non-null  str    
 4   total_cases                                 312088 non-null  float64
 5   new_cases                                   340457 non-null  float64
 6   new_cases_smoothed                          339198 non-null  float64
 7   total_deaths                                290501 non-null  float64
 8   new_deaths                                  340511 non-null  float64
 9   new_deaths_smoothed            

In [22]:
# Filter dataset for Asia (excluding non-country region entries if any exist)
asia_df = df[(df['continent'] == 'Asia') & (df['location'].notna())]

# Get unique country names
asia_countries = sorted(asia_df['location'].unique())

# Get total count
asia_country_count = len(asia_countries)

print(f"Total count of countries/territories in Asia: {asia_country_count}")
print("\nList of countries/territories in Asia:")
for country in asia_countries:
    print(f"- {country}")

Total count of countries/territories in Asia: 51

List of countries/territories in Asia:
- Afghanistan
- Armenia
- Azerbaijan
- Bahrain
- Bangladesh
- Bhutan
- Brunei
- Cambodia
- China
- Georgia
- Hong Kong
- India
- Indonesia
- Iran
- Iraq
- Israel
- Japan
- Jordan
- Kazakhstan
- Kuwait
- Kyrgyzstan
- Laos
- Lebanon
- Macao
- Malaysia
- Maldives
- Mongolia
- Myanmar
- Nepal
- North Korea
- Northern Cyprus
- Oman
- Pakistan
- Palestine
- Philippines
- Qatar
- Saudi Arabia
- Singapore
- South Korea
- Sri Lanka
- Syria
- Taiwan
- Tajikistan
- Thailand
- Timor
- Turkey
- Turkmenistan
- United Arab Emirates
- Uzbekistan
- Vietnam
- Yemen


In [23]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 1. Convert 'date' from string to datetime format (Fixing format)
df['date'] = pd.to_datetime(df['date'])

# 2. Filter the dataset to only include the continent of Asia
df_asia = df[df['continent'] == 'Asia'].copy()

# 3. Define our core metrics to audit
core_metrics = ['total_cases', 'total_deaths', 'total_vaccinations', 'icu_patients']

# 4. Calculate the percentage of MISSING data for each country
# (Formula: Missing rows / Total rows * 100)
missing_data_pct = df_asia.groupby('location')[core_metrics].apply(
    lambda x: (x.isnull().sum() / len(x)) * 100
).round(1)

# 5. Sort the list to show countries with the LEAST missing data at the top, and print the top 20
print("Percentage of Missing Data (%) per Country:")
print(missing_data_pct.sort_values(by=['icu_patients', 'total_vaccinations']).head(51))

Percentage of Missing Data (%) per Country:
                      total_cases  total_deaths  total_vaccinations  icu_patients
location                                                                         
Malaysia                      1.9           5.5                30.5           5.9
South Korea                   1.2           3.5                52.8          23.0
Israel                        3.0           5.6                38.9          29.3
Japan                         0.8           3.0                42.7          87.0
India                         2.3           5.4                29.1         100.0
Hong Kong                   100.0         100.0                32.2         100.0
Macao                       100.0         100.0                43.6         100.0
Kyrgyzstan                   39.1          39.1                46.9         100.0
China                         0.1           0.6                51.0         100.0
Singapore                     1.5          11.7       

In [24]:
# 1. Filter dataset to our 4 High-Fidelity countries
selected_countries = ['Malaysia', 'South Korea', 'Israel', 'Japan']
df_final = df_asia[df_asia['location'].isin(selected_countries)].copy()

# 2. Sort chronologically per country (Crucial before any time-series math)
df_final = df_final.sort_values(by=['location', 'date']).reset_index(drop=True)

# 3. Check and remove duplicates (Rubric 3b)
duplicates = df_final.duplicated().sum()
df_final = df_final.drop_duplicates()

# 4. Separate columns by their statistical behavior
cumulative_cols = ['total_cases', 'total_deaths', 'total_vaccinations']
snapshot_cols = ['icu_patients', 'hosp_patients', 'new_cases', 'new_deaths']

# 5. Apply Forward Fill to cumulative totals (Grouped by country so data doesn't bleed!)
df_final[cumulative_cols] = df_final.groupby('location')[cumulative_cols].ffill()

# 6. Apply Linear Interpolation to fluctuating snapshots (bridging gaps up to 14 days)
df_final[snapshot_cols] = df_final.groupby('location')[snapshot_cols].transform(
    lambda x: x.interpolate(method='linear', limit=14)
)

# 7. Fill remaining NaNs (e.g., early 2020 before vaccines/cases existed) with 0
all_core_cols = cumulative_cols + snapshot_cols
df_final[all_core_cols] = df_final[all_core_cols].fillna(0)

# 8. Final Audit
print(f"Selected Countries: {df_final['location'].unique().tolist()}")
print(f"Total Rows: {df_final.shape[0]}")
print(f"Duplicates Removed: {duplicates}")
print("\nMissing values remaining in core metrics:")
print(df_final[all_core_cols].isnull().sum())

Selected Countries: ['Israel', 'Japan', 'Malaysia', 'South Korea']
Total Rows: 5543
Duplicates Removed: 0

Missing values remaining in core metrics:
total_cases           0
total_deaths          0
total_vaccinations    0
icu_patients          0
hosp_patients         0
new_cases             0
new_deaths            0
dtype: int64


In [25]:
# 1. Generate Descriptive Statistics for our core metrics (Rubric 3b & 3c)
stats_summary = df_final[all_core_cols].describe().round(2)

# 2. Check specifically if any rows have negative (< 0) values
negative_counts = (df_final[all_core_cols] < 0).sum()

print("=== COUNT OF INVALID NEGATIVE VALUES ===")
print(negative_counts)
print("\n=== DESCRIPTIVE STATISTICS (MIN, MAX, MEAN, QUARTILES) ===")
print(stats_summary.loc[['min', 'mean', '50%', '75%', 'max']])

=== COUNT OF INVALID NEGATIVE VALUES ===
total_cases           0
total_deaths          0
total_vaccinations    0
icu_patients          0
hosp_patients         0
new_cases             0
new_deaths            0
dtype: int64

=== DESCRIPTIVE STATISTICS (MIN, MAX, MEAN, QUARTILES) ===
      total_cases  total_deaths  total_vaccinations  icu_patients  hosp_patients  new_cases  new_deaths
min          0.00          0.00        0.000000e+00          0.00           0.00        0.0        0.00
mean   6741715.89      17413.78        7.434336e+07        316.18        3836.14    14134.4       28.97
50%    1711935.00      10896.00        1.864321e+07        121.00         733.00     1252.0        6.00
75%    5052337.00      31884.00        1.196004e+08        377.00        3790.21     6710.5       26.00
max   34571873.00      74694.00        3.837477e+08       3034.00       44458.00   621328.0      592.00


In [26]:
# 1. Calculate Q1 (25%), Q3 (75%), and IQR for our core metrics (Rubric 3b)
Q1 = df_final[all_core_cols].quantile(0.25)
Q3 = df_final[all_core_cols].quantile(0.75)
IQR = Q3 - Q1

# 2. Define the standard upper boundary for outliers: Q3 + (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

# 3. Count how many rows exceed the outlier boundary in each column
outliers_count = (df_final[all_core_cols] > upper_bound).sum()

print("=== MATHEMATICAL OUTLIERS DETECTED (IQR METHOD) ===")
print(outliers_count)

# 4. Inspect the Top 10 most extreme 'new_cases' days to see WHAT they are
print("\n=== TOP 10 EXTREME 'NEW_CASES' DAYS ===")
print(df_final[['location', 'date', 'new_cases', 'icu_patients']].sort_values(by='new_cases', ascending=False).head(50))

=== MATHEMATICAL OUTLIERS DETECTED (IQR METHOD) ===
total_cases           1014
total_deaths             0
total_vaccinations     446
icu_patients           627
hosp_patients          729
new_cases              894
new_deaths             652
dtype: int64

=== TOP 10 EXTREME 'NEW_CASES' DAYS ===
         location       date  new_cases  icu_patients
4961  South Korea 2022-03-16   621328.0   1244.000000
4967  South Korea 2022-03-22   490872.0   1104.000000
4974  South Korea 2022-03-29   424573.0   1215.000000
4962  South Korea 2022-03-17   407017.0   1159.000000
4960  South Korea 2022-03-15   400730.0   1196.000000
4968  South Korea 2022-03-23   395480.0   1084.000000
4956  South Korea 2022-03-11   383651.0   1116.000000
4963  South Korea 2022-03-18   381329.0   1049.000000
4959  South Korea 2022-03-14   362322.0   1158.000000
4966  South Korea 2022-03-21   353806.0   1130.000000
4957  South Korea 2022-03-12   350169.0   1066.000000
4973  South Korea 2022-03-28   347441.0   1273.000000
495

In [27]:
# Filter raw Asian data to rows where icu_patients was ACTUALLY reported (not null)
raw_icu_valid = df_asia[df_asia['location'].isin(selected_countries) & df_asia['icu_patients'].notnull()]

# Find the first date (min), last date (max), and total reported days (count) per country
icu_window = raw_icu_valid.groupby('location')['date'].agg(['min', 'max', 'count'])

print("=== TRUE ICU REPORTING WINDOW PER COUNTRY ===")
print(icu_window)

=== TRUE ICU REPORTING WINDOW PER COUNTRY ===
                   min        max  count
location                                
Israel      2020-09-01 2023-05-07    979
Japan       2020-05-07 2023-10-18    180
Malaysia    2020-03-24 2023-10-21   1306
South Korea 2020-03-28 2023-03-04   1066


In [28]:
# 1. Include 'population' alongside our 7 core metrics for fair country comparisons
final_cols = ['location', 'date'] + all_core_cols + ['population']
df_clean = df_final[final_cols].copy()

# 2. Crop to the Common Active Surveillance Window (2020-09-01 to 2023-03-04)
start_date = '2020-09-01'
end_date = '2023-03-04'
df_clean = df_clean[(df_clean['date'] >= start_date) & (df_clean['date'] <= end_date)].copy()

# 3. Round all human count metrics to whole numbers (No fractional patients!)
df_clean[all_core_cols] = df_clean[all_core_cols].round(0)

# 4. Save the pristine dataset to a new CSV file for our future Dashboard
df_clean.to_csv('covid_cleaned.csv', index=False)

# 5. Verify final shape and check that each country has the exact same number of days
print(f"Final Cleaned Dataset Shape: {df_clean.shape}")
print("\nRecords per country (Frequency Check):")
print(df_clean['location'].value_counts())
print("\nUpdated Descriptive Statistics (Post-Cleaning):")
print(df_clean[all_core_cols].describe().round(1).loc[['min', '50%', 'mean', 'max']])

Final Cleaned Dataset Shape: (3660, 10)

Records per country (Frequency Check):
location
Israel         915
Japan          915
Malaysia       915
South Korea    915
Name: count, dtype: int64

Updated Descriptive Statistics (Post-Cleaning):
      total_cases  total_deaths  total_vaccinations  icu_patients  hosp_patients  new_cases  new_deaths
min        9340.0         127.0                 0.0           4.0            0.0        0.0         0.0
50%     1724118.5       11079.0          19990108.0         258.0         1454.0     2906.5        13.0
mean    5459155.6       16360.0          74874568.8         455.2         4960.8    20062.6        41.9
max    33241066.0       72771.0         382618059.0        3034.0        44458.0   621328.0       592.0
